# Modelo XY en 2D - Transición de Kosterlitz-Thouless

Este notebook implementa una simulación Monte Carlo del modelo XY en 2D para estudiar la transición de fase de Kosterlitz-Thouless (KT).

## Modelo XY

### Breve introducción al Modelo XY
El modelo XY es un modelo clásico de magnetismo en física estadística que consiste en una red de espines vectoriales en el plano 
(θ_i ∈ [0, 2π)), interactuando con sus vecinos más próximos. El Hamiltoniano del modelo XY es

$$\mathcal{H} = -J\sum_{<i,j>} \cos(\theta_{i} - \theta_{j})$$

Donde
  - La suma es sobre pares de primeros vecinos.
  - J es la fuerza de interacción.

En 2D el modelo XY muestra una transición de fase de Kosterlitz-Thouless a temperatura crítica KT.

En este cuaderno implementaremos una simulación del modelo XY en dos dimensiones usando el algoritmo de 
Metropolis:
- Inicializar una red cuadrada de ángulos aleatorios θ ∈ [0, 2π).
- Repetir: seleccionar un espín aleatorio, proponer un nuevo θ aleatorio, calcular la variación de energía ΔE y aceptar 
  con probabilidad min(1, exp(-ΔE / T)).
- Medir magnetización vectorial, energía y correlaciones después de un número de pasos de equilibrio.

Notas sobre la implementación:
- Usaremos Julia y la librería `Plots` para visualizar configuraciones y observables.
- Los parámetros importantes son el tamaño de la red `L`, la temperatura `T`, el número de pasos de equilibrado y el 
  número de muestras para promediar.
- El código siguiente definirá una función `xy_model()` que inicializa la red, corre la simulación y grafica resultados.

Puedes ejecutar las celdas de código para ajustar parámetros y observar la transición KT.

In [ ]:
using Plots, Random, Statistics, LaTeXStrings, DataFrames, CSV, LsqFit, Base.Threads

In [ ]:
# Directorios para animaciones y gráficos
const ANIM_DIR = "animations"
const PLOT_DIR = "plots"

# Crear directorios si no existen
mkpath(ANIM_DIR)
mkpath(PLOT_DIR)

In [ ]:
# Implementación del modelo XY usando mutable struct
mutable struct XYModel
    L::Int              # Tamaño de la red (LxL)
    J::Float64          # Fuerza de interacción
    T::Float64          # Temperatura
    spins::Matrix{Float64} # Configuración de espines (ángulos θ ∈ [0, 2π))
    M::Vector{Float64}  # Magnetización por paso
    E::Vector{Float64}  # Energía por paso
end

In [ ]:
# Función para calcular la energía local de un espín
function local_energy(spins::Matrix{Float64}, i::Int, j::Int, L::Int, J::Float64)
    θ = spins[i, j]
    neighbors = [
        spins[mod1(i-1, L), j],
        spins[mod1(i+1, L), j],
        spins[i, mod1(j-1, L)],
        spins[i, mod1(j+1, L)]
    ] # Vecinos con condiciones periódicas
    return -J * sum(cos(θ - θ_n) for θ_n in neighbors) 
end         

In [ ]:
# Función para calcular correlaciones espaciales en XY
function compute_correlations_xy(spins::Matrix{Float64}, max_r::Int=10)
    L = size(spins, 1)
    N = L * L
    cos_spins = cos.(spins)  # Precomputar cos(θ) para eficiencia
    sin_spins = sin.(spins)
    
    # Promedio de <cos(θ)> y <sin(θ)> para magnetización
    m_cos = mean(cos_spins)
    m_sin = mean(sin_spins)
    
    correlations = zeros(Float64, max_r)
    
    for r in 1:max_r
        sum_cos_diff = 0.0
        count = 0           
        
        # Suma sobre todos los pares a distancia Manhattan r
        for i in 1:L, j in 1:L
            # Dirección horizontal
            i_r = mod1(i + r, L)
            sum_cos_diff += cos_spins[i, j] * cos_spins[i_r, j] + sin_spins[i, j] * sin_spins[i_r, j]  # cos(θ_i - θ_{i+r})
            count += 1
            
            # Dirección vertical
            j_r = mod1(j + r, L)
            sum_cos_diff += cos_spins[i, j] * cos_spins[i, j_r] + sin_spins[i, j] * sin_spins[i, j_r]
            count += 1
        end
        
        # Correlación conectada: <cos(θ_i - θ_j)> - <cos(θ_i)><cos(θ_j)>
        avg_cos_diff = sum_cos_diff / count
        correlations[r] = avg_cos_diff - m_cos^2 - m_sin^2  # Aproximación para isotrópico
    end
    
    return correlations
end

In [ ]:
# Versión modular y más legible de metropolis_xy! (misma API que antes)
const two_pi = 2π

# Sumas de cos y sin de los 4 vecinos (inline y sin allocations)
@inline function neighbor_sums(spins::Matrix{Float64}, i::Int, j::Int, L::Int)
    i_m = mod1(i-1, L); i_p = mod1(i+1, L)
    j_m = mod1(j-1, L); j_p = mod1(j+1, L)
    # Evitamos crear arrays, sumamos directamente                                                                                                                                                                                                                                                           
    Sc =  cos(spins[i_m, j]) + cos(spins[i_p, j]) + cos(spins[i, j_m]) + cos(spins[i, j_p])
    Ss =  sin(spins[i_m, j]) + sin(spins[i_p, j]) + sin(spins[i, j_m]) + sin(spins[i, j_p])
    return Sc, Ss
end

# ΔE usando las sumas Sc, Ss y cos/sin ya calculados
@inline function delta_energy_from_sums(J::Float64, cos_old::Float64, sin_old::Float64, cos_new::Float64, sin_new::Float64, Sc::Float64, Ss::Float64)
    return -J * ((cos_new - cos_old) * Sc + (sin_new - sin_old) * Ss)
end

In [ ]:
function metropolis_xy!(model::XYModel, steps::Int; rng::AbstractRNG=Random.GLOBAL_RNG, animated::Bool=false, frame_interval::Int=100)
    L, J, T, spins = model.L, model.J, model.T, model.spins
    E_current = sum(local_energy(spins, i, j, L, J) for i in 1:L, j in 1:L) / 2  # Energía total inicial

    M = Vector{Float64}(undef, steps) # Magnetización
    E = Vector{Float64}(undef, steps) # Energía

    frames = animated ? Vector{Matrix{Float64}}() : nothing

    # Inicializar suma vectorial para magnetización (evita sumar toda la red cada paso)
    N = L * L
    Mx_sum = sum(cos.(spins))
    My_sum = sum(sin.(spins))

    @inbounds for step in 1:steps
        # Selección aleatoria de sitio
        i = rand(rng, 1:L)
        j = rand(rng, 1:L)

        θ_old = spins[i, j]
        cos_old = cos(θ_old); sin_old = sin(θ_old)

        # Obtener las sumas de cos/sin de los vecinos (limpia y eficiente)
        Sc, Ss = neighbor_sums(spins, i, j, L) #Suma de cosenos y senos de los vecinos

        # Propuesta (uniforme en [0,2π))
        θ_new = rand(rng) * two_pi
        cos_new, sin_new = cos(θ_new), sin(θ_new)

        # ΔE computado en O(1) usando las sumas
        ΔE = delta_energy_from_sums(J, cos_old, sin_old, cos_new, sin_new, Sc, Ss)

        # Regla de Metropolis
        if ΔE <= 0 || rand(rng) < exp(-ΔE / T)
            # Aceptar: actualizar espín, energía y sumas de magnetización
            spins[i, j] = θ_new
            E_current += ΔE
            Mx_sum += cos_new - cos_old
            My_sum += sin_new - sin_old
        end

        # Medidas (normalizadas)
        M_x = Mx_sum / N
        M_y = My_sum / N
        M[step] = sqrt(M_x^2 + M_y^2)
        E[step] = E_current / N

        if animated && (step % frame_interval == 0)
            push!(frames, copy(spins))
        end
    end

    model.M, model.E, model.spins = M, E, spins
    return model, frames
end


In [ ]:
# Función principal para simular
function xy_model(steps::Int, T::Float64, L::Int, J::Float64=1.0; rng::AbstractRNG=Random.GLOBAL_RNG)
    spins = rand(rng, L, L) .* 2π  # Configuración inicial aleatoria θ ∈ [0, 2π)
    model = XYModel(L, J, T, spins, Vector{Float64}(), Vector{Float64}()) 
    metropolis_xy!(model, steps; rng=rng) # Corre la simulación
    return model # Retorna el modelo completo con M y E
end

In [ ]:
# Paralelización usando threads
using Base.Threads

function simulate_multiple_temperatures_xy(steps::Int, L::Int, temperatures::Vector{Float64})
    results = Vector{XYModel}(undef, length(temperatures))
    @threads for i in 1:length(temperatures)
        rng = MersenneTwister(threadid() + i)  # RNG único por hilo
        results[i] = xy_model(steps, temperatures[i], L; rng=rng)
    end
    return results
end

## Animaciones

In [ ]:
# Animación de la evolución de la configuración de espines
function animate_xy(T::Float64, L::Int=20, steps::Int=10000, frame_interval::Int=100; return_anim::Bool=false, custom_title::String="")
    spins = rand(L, L) .* 2π
    model = XYModel(L, 1.0, T, spins, Vector{Float64}(), Vector{Float64}())
    model, frames = metropolis_xy!(model, steps; animated=true, frame_interval=frame_interval)

    # Crear animación con Plots: usar heatmap de cos(θ) para visualizar
    title_text = custom_title != "" ? custom_title : "XY Model at T=$T"
    anim = @animate for frame in frames
        cos_frame = cos.(frame)
        heatmap(cos_frame, title=title_text, color=:viridis, aspect_ratio=:equal, clims=(-1,1), axis=false, legend=false, ticks=false, colorbar=false, framestyle=:none)
    end

    gif_path = joinpath(ANIM_DIR, "xy_animation_T$(T).gif")
    gif(anim, gif_path, fps=5)
    return anim  # Devolver para mostrar en notebook
end

In [ ]:
anim = animate_xy(0.05, 256, 10000, 100; return_anim=true, custom_title="XY Model at Low T=0.05")

In [ ]:
# Función simple para mostrar animaciones horizontalmente (XY)
function show_animations_xy(temperatures::Vector{Float64})
    html = "<div style='display: flex; gap: 20px; justify-content: center;'>"
    for T in temperatures
        filename = "xy_animation_T$(T).gif"
        html *= "<div style='text-align:center;'><img src='" * filename * "' style='width: 900px; height: auto; display:block;'/><div style='margin-top:6px;font-size:0.9em;'>T=" * string(T) * "</div></div>"
    end
    html *= "</div>"
    display("text/html", html)
end
# Ejemplo de uso (no ejecutarlo automáticamente si prefieres hacerlo manualmente)
# contrast_temps = [0.05, 0.9, 3.0]
# animations = animate_multiple_temperatures_display_simple_xy(contrast_temps, L=64, steps=2000, frame_interval=200)
# display_animations_or_gifs(animations, contrast_temps)

In [ ]:
# Función para generar múltiples animaciones (XY) y devolverlas
function animate_multiple_temperatures_display_simple_xy(temperatures::Vector{Float64}, L::Int=20, steps::Int=3000, frame_interval::Int=200)
    # Vector para almacenar las animaciones
    animations = Vector{Any}(undef, length(temperatures))
    for i in 1:length(temperatures)
        T = temperatures[i]
        # Crear título descriptivo
        title_text = if T < 0.8
            "T=$(T) (Ordenado)"
        elseif T < 1.1
            "T=$(T) (Cercano a KT)"
        else
            "T=$(T) (Desordenado)"
        end

        # Reutilizar la función animate_xy existente (debe aceptar return_anim=true)
        anim = animate_xy(T, L, steps, frame_interval; return_anim=true, custom_title=title_text)
        animations[i] = anim
    end
    return animations
end


In [ ]:
# Intentar mostrar las animaciones directamente en la celda; si falla (entorno), mostrar los GIFs con HTML
function display_animations_or_gifs(animations::Vector{Any}, temperatures::Vector{Float64})
    try
        # Intentamos usar plot para mostrar múltiples animaciones en layout
        display(plot(animations... , layout=(1,length(animations)), size=(2400*length(animations), 900)))
    catch err
        @info "Fallo al mostrar animaciones embebidas: mostrando GIFs guardados en su lugar" err
        show_animations_xy(temperatures)
    end
end

In [ ]:
temperatures = [0.1, 1.0, 2.0]
animations = animate_multiple_temperatures_display_simple_xy(temperatures, 32, 20000, 200)

In [ ]:
display_animations_or_gifs(animations, temperatures)

## Ensembles y Análisis de Transición de Fase

In [ ]:
# Función genérica para plotear con agrupación por L (similar a Ising)
function plot_by_group_xy(df::DataFrame, x::Symbol, y::Symbol; 
                          group::Symbol=:L, 
                          xlabel="", 
                          ylabel="", 
                          posthook=nothing, 
                          savepath=nothing, 
                          fmt::Symbol=:pdf)
    groups = groupby(df, group)
    colors = palette(:tab10, length(groups))
    markers = [:circle, :square, :diamond, :utriangle, :dtriangle]
    
    p = plot(xlabel=xlabel, ylabel=ylabel, legend=:topright, grid=false, size=(900, 600))
    
    for (i, g) in enumerate(groups)
        gs = sort(g, x)
        xs, ys = gs[!, x], gs[!, y]
        label = "$group=$(unique(gs[!, group])[1])"
        scatter!(p, xs, ys; 
                marker=markers[mod1(i, length(markers))], 
                ms=6, 
                color=colors[i], 
                label=label, 
                markerstrokewidth=0)
        # Línea eliminada: ya no conectamos los puntos con plot!
    end
    
    if posthook !== nothing
        posthook(p)
    end
    
    if savepath !== nothing
        savefig(p, joinpath(PLOT_DIR, savepath * "." * string(fmt)))
    end
    
    return p
end

In [ ]:
# Función auxiliar para ajustar modelos de correlaciones usando LsqFit (estilo Ising)
function fit_correlation_model(r_data::Vector{Int}, corr_data::Vector{Float64}, model_func::Function, initial_guess::Vector{Float64})
    fit = curve_fit(model_func, r_data, corr_data, initial_guess)
    return fit.param  # Retorna los parámetros ajustados
end

In [ ]:
# Función para plotear correlaciones espaciales con fits usando LsqFit
function plot_correlations_df(df::DataFrame; 
                              L_list=nothing, 
                              T_list=[0.1, 0.89, 1.5, 3.0],  # Temperaturas representativas
                              max_r=10, 
                              savepath=nothing, 
                              fmt::Symbol=:pdf)
    
    # Si no se especifica L_list, usar todos los L únicos
    if L_list === nothing
        L_list = sort(unique(df.L))
    end
    
    n_rows = length(L_list)
    plots = []
    
    for (idx, L) in enumerate(L_list)
        p = plot(xlabel=L"Distance $r$", ylabel=L"$G(r)$", 
                title="L=$L", legend=:topright, grid=false, 
                size=(900, 600))
        
        colors = [:blue, :red, :green, :purple]
        markers = [:circle, :square, :diamond, :utriangle]
        
        for (i, T) in enumerate(T_list)
            # Filtrar datos para L y T específicos
            row = filter(r -> r.L == L && abs(r.T - T) < 0.05, df)
            if isempty(row)
                continue
            end
            
            corr = row[1, :Correlations][1:max_r]
            r_values = 1:max_r
            
            label = if T < 0.5
                latexstring("T=$(round(T, digits=2)) (Ordered)")
            elseif abs(T - 0.89) < 0.1
                latexstring("T=$(round(T, digits=2)) (Critical)")
            elseif T > 2.0
                latexstring("T=$(round(T, digits=2)) (Disordered)")
            else
                latexstring("T=$(round(T, digits=2))")
            end
            
            # Solo scatter, sin líneas conectando puntos
            scatter!(p, r_values, abs.(corr), 
                    marker=markers[i], ms=5, color=colors[i], 
                    label=label, markerstrokewidth=0)
        end
        
        # Agregar fits usando LsqFit para temperaturas seleccionadas
        r_fit = 1:max_r
        T_KT = 0.89
        
        for (i, T) in enumerate(T_list)
            row = filter(r -> r.L == L && abs(r.T - T) < 0.05, df)
            if isempty(row)
                continue
            end
            
            corr = row[1, :Correlations][1:max_r]
            r_data = Vector{Int}(1:max_r)
            corr_data = abs.(corr)
            
            # Fit para régimen desordenado (T > T_KT): G(r) ~ A * exp(-r/ξ)
            if T > 2.0
                model = (x, p) -> p[1] .* exp.(-x ./ p[2])
                initial_guess = [0.5, 1.0]
                try
                    params = fit_correlation_model(r_data, corr_data, model, initial_guess)
                    G_fitted = model(r_fit, params)
                    label_fit = "Fit: $(round(params[1], digits=2)) exp(-r/$(round(params[2], digits=2)))"
                    plot!(p, r_fit, G_fitted, label=label_fit, linestyle=:dash, color=:purple, lw=2)
                catch e
                    @warn "Fit fallido para T=$T (desordenado)" exception=e
                end
                
            # Fit para régimen crítico (T ≈ T_KT): G(r) ~ A / r^(1+η)
            elseif abs(T - T_KT) < 0.1
                model = (x, p) -> p[1] ./ (x .^ (1 + p[2]))
                initial_guess = [0.3, 0.1]
                try
                    params = fit_correlation_model(r_data, corr_data, model, initial_guess)
                    G_fitted = model(r_fit, params)
                    label_fit = "Fit: $(round(params[1], digits=2))/r^(1+$(round(params[2], digits=2)))"
                    plot!(p, r_fit, G_fitted, label=label_fit, linestyle=:dash, color=:orange, lw=2)
                catch e
                    @warn "Fit fallido para T=$T (crítico)" exception=e
                end
            end
        end
        
        push!(plots, p)
    end
    
    layout_grid = (length(plots), 1)
    p_final = plot(plots..., layout=layout_grid, size=(900, 600 * length(plots)))
    
    if savepath !== nothing
        savefig(p_final, joinpath(PLOT_DIR, savepath * "." * string(fmt)))
    end
    
    return p_final
end

In [ ]:
# Función para plotear magnetización agrupada por L
function plot_magnetization_df(df::DataFrame; savepath=nothing, fmt::Symbol=:pdf)
    T_KT = 0.89  # Temperatura crítica KT para XY 2D
    
    # Magnetización
    posthook_m = p -> vline!(p, [T_KT], label=L"$T_{KT} \approx 0.89$", linestyle=:dash, color=:black, lw=2)
    p = plot_by_group_xy(df, :T, :Magnetization; 
                         xlabel=L"Temperature $T$", 
                         ylabel=L"$\langle |M| \rangle$", 
                         posthook=posthook_m)
    
    if savepath !== nothing
        savefig(p, joinpath(PLOT_DIR, savepath * "." * string(fmt)))
    end
    
    return p
end

### Funciones de graficación modular (estilo Ising)

In [ ]:
# Función para calcular promedios de ensemble paralelos para un (L, T) específico
function ensemble_averages_single_LT(L::Int, T::Float64, n_realizations::Int, steps::Int, max_r::Int=10)
    accum_m = 0.0
    accum_corr = zeros(Float64, max_r)
    
    @threads for j in 1:n_realizations
        rng = MersenneTwister(threadid() * 1000 + j)
        model = xy_model(steps, T, L; rng=rng)
        
        # Equilibrio: últimos 1/3 de pasos
        eq_start = div(2 * length(model.M), 3) + 1
        
        m = mean(model.M[eq_start:end])
        corr = compute_correlations_xy(model.spins, max_r)
        
        # Acumular (thread-safe con @threads)
        accum_m += m
        accum_corr .+= corr
    end
    
    avg_m = accum_m / n_realizations
    avg_corr = accum_corr / n_realizations
    
    return avg_m, avg_corr
end

In [ ]:
# Función principal para generar DataFrame de ensembles (múltiples L y T)
function ensemble_multiple_sizes_xy(L_sizes::Vector{Int}, temperatures::Vector{Float64}, n_realizations::Int, steps::Int, max_r::Int=10)
    all_data = DataFrame(L=Int[], T=Float64[], Magnetization=Float64[], Correlations=Vector{Float64}[])
    
    for L in L_sizes
        println("Procesando L=$L...")
        for T in temperatures
            avg_m, avg_corr = ensemble_averages_single_LT(L, T, n_realizations, steps, max_r)
            push!(all_data, (L, T, avg_m, avg_corr))
        end
    end
    
    return all_data
end

### Ensembles para múltiples tamaños de red

Implementación estilo Ising usando DataFrames para organizar datos de ensembles con múltiples tamaños `L` y temperaturas `T`. 

**Nota**: Nos enfocamos en magnetización y correlaciones espaciales G(r), que son los observables clave para caracterizar la transición KT.

**Aviso:** Las configuraciones recomendadas arriba son intensivas en tiempo. Para ejecutar la simulación completa, activa la celda que contiene la llamada a `ensemble_multiple_sizes_xy(...)` retirando el comentario en la línea que empieza por `@time df_xy = ...` y ejecútala en un entorno con suficientes núcleos/tiempo. Si quieres, puedo crear un script separado para lanzar trabajos en background o por lotes.

In [ ]:
# Parámetros para ensembles completos (recomendados para estadística)
# Estos valores son intensivos en tiempo; no ejecutes esta celda a menos que tengas recursos y tiempo.
L_sizes = [24, 32]  # Tamaños de red — añade 48 para mejor escalamiento finito
# Muestreo de temperaturas: fino cerca de T_KT ≈ 0.89
temperatures = vcat(0.5:0.1:0.7, 0.75:0.05:1.1, 1.2:0.2:3.0)

n_realizations = 200    # Recomendado: 200-500 para buena estadística
steps = 2500000         # Pasos MC por realización (equilibrio + mediciones)
max_r = 8              # Distancias máximas para G(r) (puedes aumentar hasta L/2)

# Nota: la siguiente línea ejecuta la simulación. Ejecútala manualmente cuando estés listo.
@time df_xy = ensemble_multiple_sizes_xy(L_sizes, temperatures, n_realizations, steps, max_r)

In [ ]:
# Guardar DataFrame para análisis posterior (ejecutar después de generar df_xy)
CSV.write(joinpath(PLOT_DIR, "xy_ensemble_data.csv"), df_xy)

In [ ]:
# Plotear correlaciones espaciales con fits teóricos
# (Ejecutar después de generar `df_xy` con los parámetros recomendados)
plot_correlations_df(df_xy; 
                    L_list=[24, 32], 
                    T_list=[0.5, 0.89, 1.5, 3.0], 
                    max_r=8, 
                    savepath="xy_correlations", 
                    fmt=:png)

In [ ]:
# Plotear magnetización vs temperatura para todos los tamaños L
# (Ejecutar después de generar `df_xy` con los parámetros recomendados)
plot_magnetization_df(df_xy; savepath="xy_magnetization", fmt=:png)